In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps to UTC datetimes
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

for df in (metric, trace, log, error_logs):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Ensure numeric 'value' columns where applicable
for df in (metric, trace, log):
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Helper to create a compact string listing unique cmdb_ids (limit to 20 shown)
def compact_cmdb_list(df, limit=20):
    ids = sorted(df['cmdb_id'].dropna().unique())
    if len(ids) <= limit:
        return ", ".join(ids)
    else:
        shown = ", ".join(ids[:limit])
        return f"{shown}, ... (+{len(ids)-limit} more)"

# Helper to format min/max timestamps as ISO 8601 UTC strings (with 'Z')
def iso_utc(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(ts).tz_convert('UTC').strftime('%Y-%m-%dT%H:%M:%SZ')

# 1) metric.csv summary
metric_cmdbs = compact_cmdb_list(metric)
metric_min_ts = iso_utc(metric['timestamp'].min())
metric_max_ts = iso_utc(metric['timestamp'].max())
metric_overview = pd.DataFrame({
    "unique_cmdb_ids": [metric_cmdbs],
    "time_range_start_utc": [metric_min_ts],
    "time_range_end_utc": [metric_max_ts]
})

# For each (cmdb_id, kpi_name): count and global P95
metric_grp = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], sort=False)['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
    .reset_index()
)
metric_grp['p95_value'] = pd.to_numeric(metric_grp['p95_value'], errors='coerce').round(6)
metric_top20 = metric_grp.sort_values('count', ascending=False).head(20)[['cmdb_id', 'kpi_name', 'count', 'p95_value']].reset_index(drop=True)

# 2) trace.csv summary
trace_cmdbs = compact_cmdb_list(trace)
trace_min_ts = iso_utc(trace['timestamp'].min())
trace_max_ts = iso_utc(trace['timestamp'].max())
trace_overview = pd.DataFrame({
    "unique_cmdb_ids": [trace_cmdbs],
    "time_range_start_utc": [trace_min_ts],
    "time_range_end_utc": [trace_max_ts]
})

trace_grp = (
    trace
    .groupby(['cmdb_id', 'trace_name'], sort=False)['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
    .reset_index()
)
trace_grp['p95_value'] = pd.to_numeric(trace_grp['p95_value'], errors='coerce').round(6)
trace_top20 = trace_grp.sort_values('count', ascending=False).head(20)[['cmdb_id', 'trace_name', 'count', 'p95_value']].reset_index(drop=True)

# 3) log.csv summary
log_cmdbs = compact_cmdb_list(log)
log_min_ts = iso_utc(log['timestamp'].min())
log_max_ts = iso_utc(log['timestamp'].max())
log_overview = pd.DataFrame({
    "unique_cmdb_ids": [log_cmdbs],
    "time_range_start_utc": [log_min_ts],
    "time_range_end_utc": [log_max_ts]
})

log_grp = (
    log
    .groupby(['cmdb_id', 'log_name'], sort=False)['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
    .reset_index()
)
log_grp['p95_value'] = pd.to_numeric(log_grp['p95_value'], errors='coerce').round(6)
log_top20 = log_grp.sort_values('count', ascending=False).head(20)[['cmdb_id', 'log_name', 'count', 'p95_value']].reset_index(drop=True)

# 4) error_logs.csv summary and up to 10 distinct example messages
err_cmdbs = compact_cmdb_list(error_logs)
err_min_ts = iso_utc(error_logs['timestamp'].min())
err_max_ts = iso_utc(error_logs['timestamp'].max())
error_overview = pd.DataFrame({
    "unique_cmdb_ids": [err_cmdbs],
    "time_range_start_utc": [err_min_ts],
    "time_range_end_utc": [err_max_ts]
})

# Examples: distinct messages with timestamp and cmdb_id, limit to 10 rows
err_examples = (
    error_logs[['timestamp', 'cmdb_id', 'message']]
    .drop_duplicates(subset=['message'])
    .sort_values('timestamp')
    .head(10)
    .copy()
)
err_examples['timestamp'] = err_examples['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Final compact outputs (limited rows as requested)
metric_overview, metric_top20, trace_overview, trace_top20, log_overview, log_top20, error_overview, err_examples

```
Out[1]:
```
# summary of the previously displayed telemetry analysis
summary = (
    "Summary of telemetry (compact):\n\n"
    "Metrics:\n"
    "- Unique cmdb_ids (top seen): carts, carts-db, catalogue, catalogue-db, front-end, orders, payment, queue-master, shipping, user, session-db.\n"
    "- Time range (metrics): 2024-01-19T09:11:00Z → 2024-01-19T09:35:00Z.\n"
    "- Per-(cmdb_id,kpi) overview (top 20 by point count): most metric pairs have full series (count = 25). Notable P95 values: carts latency-90 ≈ 8.186 s, carts socket ≈ 51.81, carts mem ≈ 2.24e8, carts-db diskio ≈ 3.38e6, catalogue mem ≈ 6.57e6. These pairs are the most frequently observed metrics.\n\n"
    "Traces:\n"
    "- No trace data available (trace summary empty).\n\n"
    "Logs:\n"
    "- Unique cmdb_ids (logs) similar to metrics; time range: 2024-01-19T09:11:00Z → 2024-01-19T09:35:00Z.\n"
    "- Top log pairs are log.row_count and log.error_count for many services. log.error_count P95 is 0 for the listed services. Example log.row_count P95: front-end ≈ 1664.2, carts-db ≈ 546.8, user ≈ 520.4. session-db has only 5 points (smaller sample).\n\n"
    "Error logs:\n"
    "- No error log entries (error_logs.csv returned empty results / no example messages).\n\n"
    "Overall impression:\n"
    "- Telemetry is present for many services in metrics and logs, with no trace or textual error-log examples.\n"
    "- Hotspots to investigate further (based on P95 metrics): carts (high latency and socket values) and carts-db (high disk I/O)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                                     unique_cmdb_ids  time_range_start_utc    time_range_end_utc
0  carts, carts-db, catalogue, catalogue-db, fron...  2024-01-19T09:11:00Z  2024-01-19T09:35:00Z,          cmdb_id    kpi_name  count     p95_value
0          carts         cpu     25  1.329796e+00
1          carts       error     25  0.000000e+00
2          carts  latency-50     25  3.002634e+00
3          carts  latency-90     25  8.186146e+00
4          carts         mem     25  2.238211e+08
5          carts      socket     25  5.180667e+01
6          carts    workload     25  7.070147e+00
7       carts-db         cpu     25  4.777527e+01
8       carts-db      diskio     25  3.377773e+06
9       carts-db         mem     25  1.613920e+08
10      carts-db      socket     25  2.800000e+01
11     catalogue         cpu     25  1.518630e-01
12     catalogue      diskio     25  0.000000e+00
13     catalogue       error     25  0.000000e+00
14     catalogue  latency-50     25  3.087000e-03
15     catalogue  latency-90     25  4.756000e-03
16     catalogue         mem     25  6.565219e+06
17     catalogue      socket     25  7.000000e+00
18     catalogue    workload     25  3.474723e+00
19  catalogue-db         cpu     25  1.899260e-01,   unique_cmdb_ids time_range_start_utc time_range_end_utc
0                                 None               None, Empty DataFrame
Columns: [cmdb_id, trace_name, count, p95_value]
Index: [],                                      unique_cmdb_ids  time_range_start_utc    time_range_end_utc
0  carts, carts-db, catalogue, front-end, orders,...  2024-01-19T09:11:00Z  2024-01-19T09:35:00Z,          cmdb_id         log_name  count  p95_value
0       carts-db  log.error_count     25        0.0
1       carts-db    log.row_count     25      546.8
2      catalogue  log.error_count     25        0.0
3      catalogue    log.row_count     25      143.0
4      front-end  log.error_count     25        0.0
5      front-end    log.row_count     25     1664.2
6         orders  log.error_count     25        0.0
7         orders    log.row_count     25      108.0
8        payment  log.error_count     25        0.0
9        payment    log.row_count     25       94.0
10  queue-master  log.error_count     25        0.0
11  queue-master    log.row_count     25      486.0
12      shipping  log.error_count     25        0.0
13      shipping    log.row_count     25       54.0
14          user  log.error_count     25        0.0
15          user    log.row_count     25      520.4
16         carts  log.error_count     22        0.0
17         carts    log.row_count     22      100.0
18    session-db  log.error_count      5        0.0
19    session-db    log.row_count      5        5.0,   unique_cmdb_ids time_range_start_utc time_range_end_utc
0                                 None               None, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
# Reuse existing DataFrames if present; otherwise load (safe reload)
try:
    metric
    log
except NameError:
    metric = pd.read_csv("metric.csv")
    log = pd.read_csv("log.csv")

# Ensure timestamps are parsed to UTC datetimes (Unix seconds -> UTC)
for df in (metric, log):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# 1) Restrict analysis to these components
components = ['carts', 'catalogue', 'orders', 'payment', 'user']
metric_sel = metric[metric['cmdb_id'].isin(components)].copy()
log_sel = log[log['cmdb_id'].isin(components)].copy()

# 2) Compute global P95 for each (cmdb_id, kpi_name) using full series (metrics)
metric_p95 = (
    metric_sel
    .groupby(['cmdb_id', 'kpi_name'], sort=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)
metric_p95['global_p95'] = pd.to_numeric(metric_p95['global_p95'], errors='coerce')

# 3) Incident window filtering
start = pd.to_datetime("2024-01-19 09:08:54", utc=True)
end   = pd.to_datetime("2024-01-19 09:38:54", utc=True)
metric_window = metric_sel[(metric_sel['timestamp'] >= start) & (metric_sel['timestamp'] <= end)].copy()

# Merge per-row with global p95 so we can mark anomalies
metric_window_merged = metric_window.merge(metric_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies: value >= global_p95 (note global_p95 computed from full series)
metric_window_merged['is_anomaly'] = metric_window_merged['value'] >= metric_window_merged['global_p95']

# Aggregations per (cmdb_id, kpi_name) within window
win_agg = (
    metric_window_merged
    .groupby(['cmdb_id', 'kpi_name'], sort=False)
    .agg(
        total_points_in_window=('value', 'count'),
        anomaly_count_in_window=('is_anomaly', 'sum'),
        earliest_anomaly_ts=('timestamp', lambda s: s[s.index.isin(metric_window_merged[metric_window_merged['is_anomaly']].index)].min() if len(s)>0 else pd.NaT),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# The above earliest_anomaly_ts can be NaT; but the lambda approach above is conservative.
# A clearer way: compute earliest per-group where is_anomaly True:
anom_earliest = (
    metric_window_merged[metric_window_merged['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], sort=False)['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp': 'earliest_anomaly_ts'})
)

# Merge window agg with earliest_anomaly computed properly
win_agg = win_agg.drop(columns=['earliest_anomaly_ts']).merge(anom_earliest, on=['cmdb_id','kpi_name'], how='left')

# Now left-join full p95 table with window aggregates so we include pairs that had zero points in window
metric_summary_window = (
    metric_p95
    .merge(win_agg, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill NaNs for groups with no points in window
metric_summary_window['total_points_in_window'] = metric_summary_window['total_points_in_window'].fillna(0).astype(int)
metric_summary_window['anomaly_count_in_window'] = metric_summary_window['anomaly_count_in_window'].fillna(0).astype(int)
metric_summary_window['max_value_in_window'] = metric_summary_window['max_value_in_window'].where(metric_summary_window['total_points_in_window']>0, other=pd.NA)
metric_summary_window['earliest_anomaly_ts'] = metric_summary_window['earliest_anomaly_ts'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
metric_summary_window['earliest_anomaly_ts'] = metric_summary_window['earliest_anomaly_ts'].where(metric_summary_window['anomaly_count_in_window']>0, other=None)

# Round numeric global_p95 and max_value for compactness
metric_summary_window['global_p95'] = metric_summary_window['global_p95'].round(6)
metric_summary_window['max_value_in_window'] = pd.to_numeric(metric_summary_window['max_value_in_window'], errors='coerce').round(6)

# Sort and limit to top 20 by anomaly_count desc then max_value desc
metric_summary_window = metric_summary_window.sort_values(['anomaly_count_in_window', 'max_value_in_window'], ascending=[False, False])
metric_summary_window_top20 = metric_summary_window[['cmdb_id','kpi_name','global_p95','total_points_in_window','anomaly_count_in_window','earliest_anomaly_ts','max_value_in_window']].head(20).reset_index(drop=True)

# 4) Logs: compute global P95 across full series for (cmdb_id, log_name)
log_p95 = (
    log_sel
    .groupby(['cmdb_id','log_name'], sort=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value':'global_p95'})
)
log_p95['global_p95'] = pd.to_numeric(log_p95['global_p95'], errors='coerce')

# Filter logs to same incident window
log_window = log_sel[(log_sel['timestamp'] >= start) & (log_sel['timestamp'] <= end)].copy()

# Aggregate within window per (cmdb_id, log_name)
log_win_agg = (
    log_window
    .groupby(['cmdb_id','log_name'], sort=False)
    .agg(
        total_points_in_window=('value','count'),
        min_value_in_window=('value','min'),
        max_value_in_window=('value','max')
    )
    .reset_index()
)

# Merge global p95 with window aggregates
log_summary_window = log_p95.merge(log_win_agg, on=['cmdb_id','log_name'], how='left')

# Keep only rows where total_points_in_window > 0
log_summary_window = log_summary_window[log_summary_window['total_points_in_window'].fillna(0) > 0].copy()

# Format numeric columns and round
log_summary_window['global_p95'] = log_summary_window['global_p95'].round(6)
log_summary_window['min_value_in_window'] = pd.to_numeric(log_summary_window['min_value_in_window'], errors='coerce').round(6)
log_summary_window['max_value_in_window'] = pd.to_numeric(log_summary_window['max_value_in_window'], errors='coerce').round(6)
log_summary_window['total_points_in_window'] = log_summary_window['total_points_in_window'].astype(int)

# Sort for compact output (top by total_points then max_value) and limit to 20
log_summary_window = log_summary_window.sort_values(['total_points_in_window','max_value_in_window'], ascending=[False, False])
log_summary_window_top20 = log_summary_window[['cmdb_id','log_name','global_p95','total_points_in_window','min_value_in_window','max_value_in_window']].head(20).reset_index(drop=True)

# Final display (compact): metric_summary_window_top20, log_summary_window_top20
metric_summary_window_top20, log_summary_window_top20

```
Out[2]:
```
Summary (straightforward, based on the metric + log analysis for components: carts, catalogue, orders, payment, user; incident window = 2024-01-19T09:08:54Z → 2024-01-19T09:38:54Z):

High-level notes / caveat
- Many metric pairs (e.g., error, diskio) have global P95 = 0.0. Because anomaly was defined as value >= global_p95, those KPIs show anomaly_count = total_points_in_window (often 25) — this is not a useful anomaly signal (it’s an artifact of a zero P95 baseline). Treat those entries as uninformative unless further context confirms they are meaningful.
- log.error_count global P95 = 0 and observed values in-window are 0 → no logged errors in the window.
- Focus on non-zero P95s for meaningful deviations (memory, socket, latency, workload).

Meaningful metric signals (non-zero P95, ordered by anomaly_count / severity)
- catalogue.socket: global_p95 = 7.0 ; total_points = 25 ; anomaly_count = 25 (all points >= 7.0) ; earliest anomaly = 2024-01-19T09:11:00Z ; max = 7.0  
  (note: though non-zero, all samples meet/exceed the P95 baseline)
- payment.socket: global_p95 = 7.0 ; total_points = 25 ; anomaly_count = 25 ; earliest = 2024-01-19T09:11:00Z ; max = 7.0
- Memory spikes (each had 2 anomaly points in-window):
  - orders.mem: global_p95 = 3.302107e+08 ; total_points = 25 ; anomaly_count = 2 ; earliest_anomaly = 2024-01-19T09:12:00Z ; max ≈ 3.30218e+08
  - carts.mem:  global_p95 = 2.238211e+08 ; total_points = 25 ; anomaly_count = 2 ; earliest_anomaly = 2024-01-19T09:22:00Z ; max ≈ 2.238501e+08
  - user.mem:   global_p95 = 9.289673e+06 ; total_points = 25 ; anomaly_count = 2 ; earliest_anomaly = 2024-01-19T09:34:00Z ; max ≈ 9.321677e+06
  - catalogue.mem: global_p95 = 6.565219e+06 ; total_points = 25 ; anomaly_count = 2 ; earliest_anomaly = 2024-01-19T09:33:00Z ; max ≈ 6.566161e+06
  - payment.mem: global_p95 = 4.487782e+06 ; total_points = 25 ; anomaly_count = 2 ; earliest_anomaly = 2024-01-19T09:30:00Z ; max ≈ 4.491971e+06
- Socket and latency notable entries:
  - carts.socket: global_p95 = 51.80667 ; total_points = 25 ; anomaly_count = 2 ; earliest_anomaly = 2024-01-19T09:15:00Z ; max ≈ 53.31667
  - user.socket:  global_p95 = 38.37333 ; total_points = 25 ; anomaly_count = 2 ; earliest_anomaly = 2024-01-19T09:34:00Z ; max = 39.0
  - orders.socket: global_p95 = 24.82667 ; total_points = 25 ; anomaly_count = 2 ; earliest_anomaly = 2024-01-19T09:28:00Z ; max ≈ 25.35
  - carts latency-90: global_p95 = 8.186146s ; total_points = 25 ; anomaly_count = 2 ; earliest_anomaly = 2024-01-19T09:16:00Z ; max ≈ 8.303933s
- Workload:
  - user.workload: global_p95 ≈ 15.78699 ; total_points = 25 ; anomaly_count = 2 ; earliest_anomaly = 2024-01-19T09:19:00Z ; max ≈ 16.33777

Logs (within window; global P95 computed across full series)
- log.row_count (global P95) and in-window min/max:
  - user:   global_p95 = 520.4 ; points in window = 25 ; min = 18 ; max = 522
  - catalogue: global_p95 = 143.0 ; points = 25 ; min = 13 ; max = 147
  - orders: global_p95 = 108.0 ; points = 25 ; min = 8 ; max = 112
  - payment: global_p95 = 94.0 ; points = 25 ; min = 8 ; max = 96
  - carts:  global_p95 = 100.0 ; points in window = 22 (fewer samples) ; min = 25 ; max = 100
- log.error_count:
  - For catalogue, orders, payment, user, carts: global_p95 = 0.0 and observed min/max = 0 → no errors recorded in-window.

Actionable summary
- Check memory spikes across services (orders, carts, user, catalogue, payment): each had 2 points above their global P95 in the incident window — investigate what operations occurred around the earliest anomaly times (09:12–09:34 UTC range).
- Investigate carts specifically for both latency and socket activity (latency-90 and socket show spikes around 09:15–09:16 and 09:22). carts also has memory anomalies — carts is a high-priority candidate.
- Orders and user also show elevated socket and memory values — check service-to-database connectivity and resource usage.
- Logs do not show increased error_count; log.row_count shows normal/expected traffic volumes (user row_count is high in absolute terms).
- Treat the many “anomalies” where global_p95 == 0 as non-actionable/noisy unless you confirm those KPIs should not be zero.

If you want, I can:
- produce the filtered per-point anomaly rows for any specific KPI to inspect exact timestamps/values, or
- re-compute anomalies using a different rule (e.g., value > global_p95 and global_p95 > 0) to suppress the zero-P95 noise.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name    global_p95  total_points_in_window  anomaly_count_in_window   earliest_anomaly_ts  max_value_in_window
0   catalogue      socket  7.000000e+00                      25                       25  2024-01-19T09:11:00Z         7.000000e+00
1     payment      socket  7.000000e+00                      25                       25  2024-01-19T09:11:00Z         7.000000e+00
2       carts       error  0.000000e+00                      25                       25  2024-01-19T09:11:00Z         0.000000e+00
3   catalogue      diskio  0.000000e+00                      25                       25  2024-01-19T09:11:00Z         0.000000e+00
4   catalogue       error  0.000000e+00                      25                       25  2024-01-19T09:11:00Z         0.000000e+00
5      orders       error  0.000000e+00                      25                       25  2024-01-19T09:11:00Z         0.000000e+00
6     payment      diskio  0.000000e+00                      25                       25  2024-01-19T09:11:00Z         0.000000e+00
7     payment       error  0.000000e+00                      25                       25  2024-01-19T09:11:00Z         0.000000e+00
8        user      diskio  0.000000e+00                      25                       25  2024-01-19T09:11:00Z         0.000000e+00
9        user       error  0.000000e+00                      25                       25  2024-01-19T09:11:00Z         0.000000e+00
10     orders         mem  3.302107e+08                      25                        2  2024-01-19T09:12:00Z         3.302180e+08
11      carts         mem  2.238211e+08                      25                        2  2024-01-19T09:22:00Z         2.238501e+08
12       user         mem  9.289673e+06                      25                        2  2024-01-19T09:34:00Z         9.321677e+06
13  catalogue         mem  6.565219e+06                      25                        2  2024-01-19T09:33:00Z         6.566161e+06
14    payment         mem  4.487782e+06                      25                        2  2024-01-19T09:30:00Z         4.491971e+06
15      carts      socket  5.180667e+01                      25                        2  2024-01-19T09:15:00Z         5.331667e+01
16       user      socket  3.837333e+01                      25                        2  2024-01-19T09:34:00Z         3.900000e+01
17     orders      socket  2.482667e+01                      25                        2  2024-01-19T09:28:00Z         2.535000e+01
18       user    workload  1.578699e+01                      25                        2  2024-01-19T09:19:00Z         1.633777e+01
19      carts  latency-90  8.186146e+00                      25                        2  2024-01-19T09:16:00Z         8.303933e+00,      cmdb_id         log_name  global_p95  total_points_in_window  min_value_in_window  max_value_in_window
0       user    log.row_count       520.4                      25                   18                  522
1  catalogue    log.row_count       143.0                      25                   13                  147
2     orders    log.row_count       108.0                      25                    8                  112
3    payment    log.row_count        94.0                      25                    8                   96
4  catalogue  log.error_count         0.0                      25                    0                    0
5     orders  log.error_count         0.0                      25                    0                    0
6    payment  log.error_count         0.0                      25                    0                    0
7       user  log.error_count         0.0                      25                    0                    0
8      carts    log.row_count       100.0                      22                   25                  100
9      carts  log.error_count         0.0                      22                    0                    0)```
```

In [ ]:
# In[3]:

import pandas as pd

# Ensure metric is loaded and timestamps parsed (reuse if present)
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# Parse timestamps to UTC and ensure numeric values
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Define the exact component-KPI pairs requested
selected_pairs = {
    ('orders', 'mem'),
    ('carts', 'mem'),
    ('user', 'mem'),
    ('catalogue', 'mem'),
    ('payment', 'mem'),
    ('carts', 'socket'),
    ('carts', 'latency-90'),  # exact kpi_name 'latency-90'
    ('orders', 'socket'),
    ('user', 'socket'),
}

# 1) Compute global P95 for each full (cmdb_id, kpi_name) series (use entire metric series)
# Compute p95 for all combos, then filter to selected_pairs
full_p95_all = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], sort=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value':'global_p95'})
)
# Filter to only selected pairs
full_p95 = full_p95_all[full_p95_all.apply(lambda r: (r['cmdb_id'], r['kpi_name']) in selected_pairs, axis=1)].copy()
full_p95['global_p95'] = pd.to_numeric(full_p95['global_p95'], errors='coerce')

# 2) Filter metric points to the incident window
start = pd.to_datetime("2024-01-19 09:08:54", utc=True)
end   = pd.to_datetime("2024-01-19 09:38:54", utc=True)
metric_window = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end)].copy()

# Keep only rows matching our selected pairs in the window
metric_window = metric_window[metric_window.apply(lambda r: (r['cmdb_id'], r['kpi_name']) in selected_pairs, axis=1)].copy()

# 3) Merge global_p95 (calculated on full series) into window rows, then filter where value >= global_p95 AND global_p95 > 0
metric_window_merged = metric_window.merge(full_p95, on=['cmdb_id','kpi_name'], how='left')

# Keep only rows where global_p95 is present and > 0
metric_window_merged = metric_window_merged[metric_window_merged['global_p95'].notna()]
metric_window_merged = metric_window_merged[metric_window_merged['global_p95'] > 0]

# Filter anomalies: value >= global_p95
anomalies = metric_window_merged[metric_window_merged['value'] >= metric_window_merged['global_p95']].copy()

# Prepare output columns and formatting
anomalies['timestamp_iso_utc'] = anomalies['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
anomalies['value'] = anomalies['value'].round(6)
anomalies['global_p95'] = anomalies['global_p95'].round(6)

anomalies_df = anomalies[['timestamp_iso_utc','cmdb_id','kpi_name','value','global_p95']].sort_values('timestamp_iso_utc', ascending=True).head(50).reset_index(drop=True)

# Display the anomalies_df (compact, up to 50 rows)
anomalies_df

```
Out[3]:
```
Short summary of findings (selected KPIs, incident window 2024-01-19T09:08:54Z → 2024-01-19T09:38:54Z):

- Total anomalies found: 18 rows (each returned row is a metric point where value >= the global P95 and global P95 > 0).
- All 9 requested KPI series produced 2 anomaly points each (9 KPIs × 2 = 18).

Earliest / latest
- Earliest anomaly: 2024-01-19T09:12:00Z (orders.mem).
- Latest anomalies: 2024-01-19T09:35:00Z (multiple services/KPIs).

Per-KPI highlights (timestamp, observed value, global P95 — rounded):
- orders.mem: 
  - 2024-01-19T09:12:00Z value ≈ 3.302121e+08, global_p95 ≈ 3.302107e+08
  - 2024-01-19T09:29:00Z value ≈ 3.302180e+08, global_p95 ≈ 3.302107e+08
- carts.mem:
  - 2024-01-19T09:22:00Z value ≈ 2.238286e+08, global_p95 ≈ 2.238211e+08
  - 2024-01-19T09:35:00Z value ≈ 2.238501e+08, global_p95 ≈ 2.238211e+08
- user.mem:
  - 2024-01-19T09:34:00Z value ≈ 9.292459e+06, global_p95 ≈ 9.289673e+06
  - 2024-01-19T09:35:00Z value ≈ 9.321677e+06, global_p95 ≈ 9.289673e+06
- catalogue.mem:
  - 2024-01-19T09:33:00Z value ≈ 6.566161e+06, global_p95 ≈ 6.565219e+06
  - 2024-01-19T09:35:00Z value ≈ 6.565888e+06, global_p95 ≈ 6.565219e+06
- payment.mem:
  - 2024-01-19T09:30:00Z value ≈ 4.487851e+06, global_p95 ≈ 4.487782e+06
  - 2024-01-19T09:35:00Z value ≈ 4.491971e+06, global_p95 ≈ 4.487782e+06
- carts.socket:
  - 2024-01-19T09:15:00Z value ≈ 51.90, global_p95 ≈ 51.80667
  - 2024-01-19T09:25:00Z value ≈ 53.31667, global_p95 ≈ 51.80667
- carts (latency-90):
  - 2024-01-19T09:16:00Z value ≈ 8.303933 s, global_p95 ≈ 8.186146 s
  - 2024-01-19T09:25:00Z value ≈ 8.196092 s, global_p95 ≈ 8.186146 s
- orders.socket:
  - 2024-01-19T09:28:00Z value ≈ 25.35, global_p95 ≈ 24.82667
  - 2024-01-19T09:31:00Z value = 25.00, global_p95 ≈ 24.82667
- user.socket:
  - 2024-01-19T09:34:00Z value ≈ 38.96667, global_p95 ≈ 38.37333
  - 2024-01-19T09:35:00Z value = 39.00, global_p95 ≈ 38.37333

Interpretation / next steps
- Short-lived spikes occurred across memory, socket, and latency KPIs for the selected services during the window. Memory spikes are present for orders, carts, user, catalogue and payment; socket/latency spikes affect carts, orders, and user.
- Recommended next steps: inspect service logs and traces around the listed timestamps (09:12 → 09:35 UTC) for the affected services (especially carts and orders), check host-level resource usage and network/connectivity around those times, and correlate with any higher-level incidents or load changes.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp_iso_utc    cmdb_id    kpi_name         value    global_p95
0   2024-01-19T09:12:00Z     orders         mem  3.302121e+08  3.302107e+08
1   2024-01-19T09:15:00Z      carts      socket  5.190000e+01  5.180667e+01
2   2024-01-19T09:16:00Z      carts  latency-90  8.303933e+00  8.186146e+00
3   2024-01-19T09:22:00Z      carts         mem  2.238286e+08  2.238211e+08
4   2024-01-19T09:25:00Z      carts  latency-90  8.196092e+00  8.186146e+00
5   2024-01-19T09:25:00Z      carts      socket  5.331667e+01  5.180667e+01
6   2024-01-19T09:28:00Z     orders      socket  2.535000e+01  2.482667e+01
7   2024-01-19T09:29:00Z     orders         mem  3.302180e+08  3.302107e+08
8   2024-01-19T09:30:00Z    payment         mem  4.487851e+06  4.487782e+06
9   2024-01-19T09:31:00Z     orders      socket  2.500000e+01  2.482667e+01
10  2024-01-19T09:33:00Z  catalogue         mem  6.566161e+06  6.565219e+06
11  2024-01-19T09:34:00Z       user         mem  9.292459e+06  9.289673e+06
12  2024-01-19T09:34:00Z       user      socket  3.896667e+01  3.837333e+01
13  2024-01-19T09:35:00Z      carts         mem  2.238501e+08  2.238211e+08
14  2024-01-19T09:35:00Z  catalogue         mem  6.565888e+06  6.565219e+06
15  2024-01-19T09:35:00Z    payment         mem  4.491971e+06  4.487782e+06
16  2024-01-19T09:35:00Z       user         mem  9.321677e+06  9.289673e+06
17  2024-01-19T09:35:00Z       user      socket  3.900000e+01  3.837333e+01```
```